In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from alibi.explainers import CEM
from XAIEvaluator import XAIEvaluator
from MLEvaluator import MLEvaluator

In [1]:
import pickle

with open('evaluator.pkl', 'rb') as f:
    evaluator = pickle.load(f)

print("evaluator chargé ✓")

evaluator chargé ✓


In [3]:
evaluator = MLEvaluator.load("evaluator.pkl")

Evaluator chargé depuis 'evaluator.pkl'
   Modèles disponibles : ['SVM', 'Logistic_Regression', 'Decision_Tree', 'KNN', 'Naive_Bayes', 'Random_Forest', 'Extra_Trees', 'Bagging', 'Gradient_Boosting', 'MLP', 'XGBoost']


In [4]:
# ============================================================
# CLASSE CEMEvaluator
# ============================================================
class MLPWrapper(nn.Module):
    """
    Réseau de neurones PyTorch qui imite un modèle sklearn.
    
    CEM a besoin de calculer des gradients pour trouver PP et PN.
    Les modèles sklearn (RF, XGBoost, DT) ne supportent pas les gradients car ce sont des arbres de décision (fonctions en escalier).
    
    Ce MLP apprend à reproduire les probabilités du modèle sklearn, et fournit à CEM un modèle différentiable sur lequel calculer des gradients.
    
    Architecture : n_features → 128 → 64 → n_classes
    """
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 128),  # 184 → 128
            nn.ReLU(),                   # activation non-linéaire
            nn.Dropout(0.2),             # évite le surapprentissage
            nn.Linear(128, 64),          # 128 → 64
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),    # 64 → 4 classes
            nn.Softmax(dim=1)            # sortie = probabilités (somme = 1)
        )

    def forward(self, x):
        return self.network(x)
    


In [5]:
class Autoencoder(nn.Module):
    """
    Autoencoder PyTorch en forme de sablier.
    
    CEM sans autoencoder peut proposer des PP/PN avec des valeurs
    aberrantes qui n'existent pas dans la réalité.
    L'autoencoder apprend la "forme" des vrais étudiants sur X_train,
    et pénalise CEM si ses propositions s'en éloignent trop.
    
    Architecture sablier :
    Encodeur : 184 → 64 → 32  (compression)
    Décodeur : 32  → 64 → 184 (reconstruction)
    """
    def __init__(self, n_features):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, n_features),
            nn.Sigmoid()  # sortie entre 0 et 1 (données normalisées)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [6]:
class CEMEvaluator(XAIEvaluator):
    def __init__(self, ml_evaluator):
        super().__init__(ml_evaluator)  # récupère tout de XAIEvaluator

        # class_mapping (comme SHAP et DiCE)
        unique_labels = sorted(self.y_test.unique())
        self.class_mapping = {
            code: self.CLASS_NAMES[i]
            for i, code in enumerate(unique_labels)
        }

        # Infos sur les dimensions
        self.n_features = len(self.feature_names)
        self.n_classes  = len(self.CLASS_NAMES)

        # Bornes des features pour CEM
        self.feature_range = (
            self.X_train.values.min(axis=0).astype(np.float32),
            self.X_train.values.max(axis=0).astype(np.float32)
        )

        # Tiroirs propres à CEM
        self.mlp_wrappers   = {}   # un MLP wrapper par modèle
        self.autoencoder    = None # un seul autoencoder commun
        self.cem_results    = {}   # PP + PN par modèle
        self.cem_metrics    = {}   # stabilité + fidélité
        
    def _train_mlp_wrapper(self, model, model_name, epochs=100, batch_size=32):
        """
        Entraîne le MLP à imiter les probabilités du modèle sklearn.
        
        1. sklearn prédit des probabilités sur X_train
        2. Le MLP apprend à reproduire ces probabilités
        """
        print(f"\n   Entraînement MLP wrapper pour {model_name}...")
    
        # Étape 1 — Récupérer les probabilités du modèle sklearn
        X_np   = self.X_train.values.astype(np.float32)
        y_prob = model.predict_proba(self.X_train).astype(np.float32)
        # y_prob shape : (n_samples, 4) — une proba par classe
    
        # Étape 2 — Convertir en tenseurs PyTorch
        X_tensor = torch.FloatTensor(X_np)
        y_tensor = torch.FloatTensor(y_prob)
    
        # Étape 3 — DataLoader pour entraîner par batchs
        dataset    = TensorDataset(X_tensor, y_tensor)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
        # Étape 4 — Créer le MLP et l'optimiseur
        mlp       = self._MLPWrapper(self.n_features, self.n_classes)
        optimizer = optim.Adam(mlp.parameters(), lr=0.001)
        criterion = nn.MSELoss()  # MSE car on imite des probabilités continues
    
        # Étape 5 — Boucle d'entraînement
        mlp.train()
        for epoch in range(epochs):
            total_loss = 0
            for X_batch, y_batch in dataloader:
                optimizer.zero_grad()   # remettre les gradients à zéro
                output = mlp(X_batch)   # prédiction du MLP
                loss   = criterion(output, y_batch)  # écart avec sklearn
                loss.backward()         # calculer les gradients
                optimizer.step()        # ajuster les poids
                total_loss += loss.item()
    
            if (epoch + 1) % 20 == 0:
                avg_loss = total_loss / len(dataloader)
                print(f"      Epoch {epoch+1}/{epochs} — Loss : {avg_loss:.4f}")
    
        mlp.eval()
        print(f"   MLP wrapper entraîné pour {model_name} ✓")
        return mlp


    def build_mlp_wrappers(self, epochs=100):
        """
        Entraîne un MLP wrapper pour chaque modèle sélectionné.
        """
        print("\n CONSTRUCTION DES MLP WRAPPERS")
        print("=" * 60)

        for model_name, model in self.trained_models.items():
            try:
                mlp = self._train_mlp_wrapper(model, model_name, epochs=epochs)
                self.mlp_wrappers[model_name] = mlp
            except Exception as e:
                print(f"   Erreur MLP wrapper pour {model_name} : {e}")
                continue

        print(f"\n   MLP wrappers construits pour {len(self.mlp_wrappers)} modèles")
        

    def build_mlp_wrappers(self, epochs=100):
        """
        Entraîne un MLP wrapper pour chaque modèle sélectionné.
        
        Pour chaque modèle sklearn :
        1. Récupérer les probabilités predict_proba sur X_train
        2. Entraîner le MLP à reproduire ces probabilités
        3. Sauvegarder dans self.mlp_wrappers
        
        Args:
            epochs : nombre d'epochs d'entraînement par wrapper
        """
        print("\n CONSTRUCTION DES MLP WRAPPERS")
        print("=" * 60)
    
        for model_name, model in self.trained_models.items():
            print(f"\n   Entraînement MLP wrapper pour {model_name}...")
    
            # Récupérer les probabilités du modèle sklearn sur X_train
            X_np   = self.X_train.values.astype(np.float32)
            y_prob = model.predict_proba(self.X_train).astype(np.float32)
            # y_prob shape : (n_samples, 4) — une proba par classe
    
            # Convertir en tenseurs PyTorch
            X_tensor = torch.FloatTensor(X_np)
            y_tensor = torch.FloatTensor(y_prob)
    
            # DataLoader pour entraîner par batchs
            dataset    = TensorDataset(X_tensor, y_tensor)
            dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
            # Créer le MLP et l'optimiseur
            mlp       = MLPWrapper(self.n_features, self.n_classes)
            optimizer = optim.Adam(mlp.parameters(), lr=0.001)
            criterion = nn.MSELoss()  # MSE car on imite des probabilités continues
    
            # Boucle d'entraînement
            mlp.train()
            for epoch in range(epochs):
                total_loss = 0
                for X_batch, y_batch in dataloader:
                    optimizer.zero_grad()        # remettre les gradients à zéro
                    output = mlp(X_batch)        # prédiction du MLP
                    loss   = criterion(output, y_batch)  # écart avec sklearn
                    loss.backward()              # calculer les gradients
                    optimizer.step()             # ajuster les poids
                    total_loss += loss.item()
    
                if (epoch + 1) % 20 == 0:
                    avg_loss = total_loss / len(dataloader)
                    print(f"      Epoch {epoch+1}/{epochs} — Loss : {avg_loss:.4f}")
    
            mlp.eval()
            self.mlp_wrappers[model_name] = mlp
            print(f"   MLP wrapper entraîné pour {model_name} ✓")
    
        print(f"\n   MLP wrappers construits pour {len(self.mlp_wrappers)} modèles")
    
    def build_autoencoder(self, epochs=100, batch_size=32):
        """
        Entraîne l'autoencoder sur X_train normalisé.

        Étapes :
        1. Normaliser X_train entre 0 et 1 (obligatoire pour la Sigmoid)
        2. Entraîner l'autoencoder à reconstruire les données
        3. Sauvegarder les paramètres de normalisation
           (on en aura besoin plus tard quand CEM passera des données)
        """
        print("\n CONSTRUCTION DE L'AUTOENCODER")
        print("=" * 60)

        # Étape 1 — Normaliser X_train entre 0 et 1
        X_np    = self.X_train.values.astype(np.float32)
        X_min   = X_np.min(axis=0)   # minimum de chaque feature
        X_max   = X_np.max(axis=0)   # maximum de chaque feature
        X_range = X_max - X_min
        # Éviter division par zéro sur les colonnes constantes
        X_range[X_range == 0] = 1
        X_norm  = (X_np - X_min) / X_range  # valeurs entre 0 et 1

        # Sauvegarder les paramètres de normalisation
        # CEM enverra des données brutes → on doit les normaliser
        # avant de les passer à l'autoencoder
        self._ae_min   = X_min
        self._ae_max   = X_max
        self._ae_range = X_range

        # Étape 2 — Préparer le DataLoader
        X_tensor   = torch.FloatTensor(X_norm)
        dataset    = TensorDataset(X_tensor)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        # Étape 3 — Créer l'autoencoder et l'optimiseur
        ae        = Autoencoder(self.n_features)
        optimizer = optim.Adam(ae.parameters(), lr=0.001)
        criterion = nn.MSELoss()  # écart entre entrée et reconstruction

        # Étape 4 — Boucle d'entraînement
        ae.train()
        for epoch in range(epochs):
            total_loss = 0
            for (X_batch,) in dataloader:
                optimizer.zero_grad()
                output = ae(X_batch)                 # reconstruire l'entrée
                loss   = criterion(output, X_batch)  # comparer avec l'original
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            if (epoch + 1) % 20 == 0:
                avg_loss = total_loss / len(dataloader)
                print(f"   Epoch {epoch+1}/{epochs} — Loss reconstruction : {avg_loss:.4f}")

        ae.eval()
        self.autoencoder = ae
        print("   Autoencoder entraîné ✓")

    def _get_mlp_predict_fn(self, mlp, X):
        """
        Traduit le MLP PyTorch en fonction compatible Alibi.
        Alibi attend : fonction(X numpy) → probabilités numpy
        """
        X_tensor = torch.FloatTensor(X.astype(np.float32))
        with torch.no_grad():
            probs = mlp(X_tensor).numpy()
        return probs

    def _get_ae_predict_fn(self, X):
        """
        Traduit l'autoencoder PyTorch en fonction compatible Alibi.
        Alibi attend : fonction(X numpy) → X reconstruit numpy
        """
        X_norm   = (X.astype(np.float32) - self._ae_min) / self._ae_range
        X_tensor = torch.FloatTensor(X_norm)
        with torch.no_grad():
            reconstructed = self.autoencoder(X_tensor).numpy()
        return reconstructed * self._ae_range + self._ae_min

    def generate_cem_explanations(self, kappa=0.1, beta=0.1, gamma=100,
                                   max_iterations=500, learning_rate_init=0.01):
        """
        Génère les explications CEM (PP et PN) pour chaque modèle
        et chaque échantillon sélectionné par select_samples().

        Args:
            kappa             : marge de confiance entre classe prédite et 2e
                                Plus élevé = plus robuste mais plus dur à trouver
            beta              : poids régularisation L1 (sparsité)
                                Plus élevé = moins de features impliquées
            gamma             : poids de l'autoencoder
                                Plus élevé = explications plus réalistes
            max_iterations    : iterations max par optimisation
            learning_rate_init: taux d'apprentissage
        """
        print("\n GÉNÉRATION DES EXPLICATIONS CEM")
        print("=" * 60)

        if not self.mlp_wrappers:
            print("   Lance d'abord build_mlp_wrappers()")
            return
        if self.autoencoder is None:
            print("   Lance d'abord build_autoencoder()")
            return
        if not self.selected_samples:
            print("   Lance d'abord select_samples()")
            return

        shape = (1, self.n_features)

        for model_name, mlp in self.mlp_wrappers.items():
            print(f"\n   Modèle : {model_name}")
            print("-" * 40)

            model_results = {}

            for class_name, sample_indices in self.selected_samples.items():
                if not sample_indices:
                    print(f"   Aucun échantillon pour {class_name}")
                    continue

                print(f"\n   Classe : {class_name}")
                class_results = []

                for sample_idx in sample_indices:
                    sample = self.X_test.iloc[[sample_idx]].values.astype(np.float32)

                    # Prédiction du vrai modèle sklearn
                    sklearn_model     = self.trained_models[model_name]
                    predicted_encoded = int(sklearn_model.predict(
                        self.X_test.iloc[[sample_idx]]
                    )[0])
                    predicted_class = self.class_mapping.get(predicted_encoded)
                    true_encoded    = int(self.y_test.iloc[sample_idx])
                    true_class      = self.class_mapping.get(true_encoded)
                    correct         = "OK" if true_encoded == predicted_encoded else "KO"

                    print(f"\n      Échantillon {sample_idx} {correct} "
                          f"— Vraie : {true_class} | Prédite : {predicted_class}")

                    # Tiroir pour stocker les résultats de cet échantillon
                    result = {
                        "sample_idx"      : sample_idx,
                        "true_class"      : true_class,
                        "predicted_class" : predicted_class,
                        "correct"         : correct,
                        "PP"              : None,
                        "PN"              : None,
                        "PP_features"     : None,
                        "PN_features"     : None
                    }

                    # ---- PP ----
                    try:
                        cem_pp = CEM(
                            self._get_mlp_predict_fn,
                            mode='PP',
                            shape=shape,
                            kappa=kappa,
                            beta=beta,
                            gamma=gamma,
                            ae_model=self._get_ae_predict_fn,
                            max_iterations=max_iterations,
                            learning_rate_init=learning_rate_init,
                            feature_range=self.feature_range,
                            verbose=False
                        )
                        cem_pp.fit(
                            self.X_train.values.astype(np.float32),
                            no_info_type='median'
                        )
                        exp_pp = cem_pp.explain(sample)

                        if exp_pp.PP is not None:
                            pp_values = exp_pp.PP.flatten()
                            # Features non nulles = features pertinentes
                            pp_mask     = np.abs(pp_values) > 1e-6
                            pp_features = {
                                self.feature_names[i]: float(pp_values[i])
                                for i in range(self.n_features) if pp_mask[i]
                            }
                            result["PP"]          = pp_values
                            result["PP_features"] = pp_features
                            print(f"      PP : {len(pp_features)} features ✓")
                        else:
                            print(f"      PP : non trouvé")

                    except Exception as e:
                        print(f"      Erreur PP échantillon {sample_idx} : {e}")

                    # ---- PN ----
                    try:
                        cem_pn = CEM(
                            self._get_mlp_predict_fn,
                            mode='PN',
                            shape=shape,
                            kappa=kappa,
                            beta=beta,
                            gamma=gamma,
                            ae_model=self._get_ae_predict_fn,
                            max_iterations=max_iterations,
                            learning_rate_init=learning_rate_init,
                            feature_range=self.feature_range,
                            verbose=False
                        )
                        cem_pn.fit(
                            self.X_train.values.astype(np.float32),
                            no_info_type='median'
                        )
                        exp_pn = cem_pn.explain(sample)

                        if exp_pn.PN is not None:
                            pn_values = exp_pn.PN.flatten()
                            pn_mask     = np.abs(pn_values) > 1e-6
                            pn_features = {
                                self.feature_names[i]: float(pn_values[i])
                                for i in range(self.n_features) if pn_mask[i]
                            }
                            result["PN"]          = pn_values
                            result["PN_features"] = pn_features
                            print(f"      PN : {len(pn_features)} features ✓")
                        else:
                            print(f"      PN : non trouvé")

                    except Exception as e:
                        print(f"      Erreur PN échantillon {sample_idx} : {e}")

                    class_results.append(result)

                model_results[class_name] = class_results

            self.cem_results[model_name] = model_results

        print(f"\n   Explications CEM générées pour {len(self.cem_results)} modèles")

    def plot_cem_explanations(self, model_name, max_features=15):
        """
        Visualise les explications CEM pour un modèle donné.

        Pour chaque échantillon :
        - Barplot PP en bleu   → features qui CONFIRMENT la prédiction
        - Barplot PN en orange → features qui DIFFÉRENCIENT la prédiction
        Les deux côte à côte pour faciliter la comparaison.

        Args:
            model_name   : nom du modèle à visualiser
            max_features : nombre maximum de features à afficher
        """
        if model_name not in self.cem_results:
            print(f"   Pas de résultats CEM pour {model_name}")
            return

        print(f"\n GRAPHIQUES CEM — {model_name}")
        print("=" * 60)

        for class_name, class_results in self.cem_results[model_name].items():
            if not class_results:
                continue

            print(f"\n   Classe : {class_name}")

            for result in class_results:
                sample_idx      = result["sample_idx"]
                true_class      = result["true_class"]
                predicted_class = result["predicted_class"]
                correct         = result["correct"]
                pp_features     = result["PP_features"]
                pn_features     = result["PN_features"]

                has_pp = pp_features is not None and len(pp_features) > 0
                has_pn = pn_features is not None and len(pn_features) > 0

                if not has_pp and not has_pn:
                    print(f"   Échantillon {sample_idx} : aucune explication disponible")
                    continue

                # Nombre de graphiques selon ce qui est disponible
                n_plots = int(has_pp) + int(has_pn)
                fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 7))
                if n_plots == 1:
                    axes = [axes]

                ax_idx = 0

                # ---- PP ----
                if has_pp:
                    ax = axes[ax_idx]
                    ax_idx += 1

                    # Trier par valeur absolue décroissante, top max_features
                    sorted_pp   = sorted(
                        pp_features.items(),
                        key=lambda x: abs(x[1]),
                        reverse=True
                    )[:max_features]

                    features_pp = [f for f, _ in sorted_pp]
                    values_pp   = [v for _, v in sorted_pp]
                    colors_pp   = ["#2196F3" if v >= 0 else "#90CAF9" for v in values_pp]

                    bars = ax.barh(features_pp, values_pp, color=colors_pp, alpha=0.85)
                    ax.axvline(x=0, color="black", linewidth=0.8)
                    ax.set_title(
                        f"PP — Pertinent Positives\n"
                        f"Features qui CONFIRMENT : {predicted_class}",
                        fontsize=11, fontweight="bold", color="#2196F3"
                    )
                    ax.set_xlabel("Valeur de la feature")
                    ax.grid(True, alpha=0.3, axis="x")

                    for bar, val in zip(bars, values_pp):
                        ax.text(
                            bar.get_width() + abs(max(values_pp, key=abs)) * 0.02,
                            bar.get_y() + bar.get_height() / 2,
                            f"{val:.3f}", va="center", fontsize=8
                        )

                # ---- PN ----
                if has_pn:
                    ax = axes[ax_idx]

                    sorted_pn   = sorted(
                        pn_features.items(),
                        key=lambda x: abs(x[1]),
                        reverse=True
                    )[:max_features]

                    features_pn = [f for f, _ in sorted_pn]
                    values_pn   = [v for _, v in sorted_pn]
                    colors_pn   = ["#FF5722" if v >= 0 else "#FFCCBC" for v in values_pn]

                    bars = ax.barh(features_pn, values_pn, color=colors_pn, alpha=0.85)
                    ax.axvline(x=0, color="black", linewidth=0.8)
                    ax.set_title(
                        f"PN — Pertinent Negatives\n"
                        f"Features qui DIFFÉRENCIENT : {predicted_class}",
                        fontsize=11, fontweight="bold", color="#FF5722"
                    )
                    ax.set_xlabel("Valeur de la feature")
                    ax.grid(True, alpha=0.3, axis="x")

                    for bar, val in zip(bars, values_pn):
                        ax.text(
                            bar.get_width() + abs(max(values_pn, key=abs)) * 0.02,
                            bar.get_y() + bar.get_height() / 2,
                            f"{val:.3f}", va="center", fontsize=8
                        )

                plt.suptitle(
                    f"{model_name} — Échantillon {sample_idx} {correct}\n"
                    f"Vraie classe : {true_class} | Prédiction : {predicted_class}",
                    fontsize=13, fontweight="bold"
                )
                plt.tight_layout()
                plt.show()

                print(f"   Échantillon {sample_idx} : "
                      f"PP={len(pp_features) if has_pp else 0} features | "
                      f"PN={len(pn_features) if has_pn else 0} features")

    def calculate_stability(self, n_perturbations=5):
        """
        Calcule la stabilité des explications CEM.

        Principe : on ajoute un petit bruit sur les features de l'échantillon,
        on régénère les PP, on mesure la corrélation entre PP original et PP perturbé.
        Proche de 1 = explications stables et reproductibles.

        Note : n_perturbations faible car CEM est lent à calculer.

        Args:
            n_perturbations : nombre de perturbations par échantillon
        """
        print("\n CALCUL DE LA STABILITÉ CEM")
        print("=" * 60)

        stability_results = {}

        for model_name, mlp in self.mlp_wrappers.items():
            print(f"\n   Modèle : {model_name}")
            model_stab = {}

            for class_name, class_results in self.cem_results.get(model_name, {}).items():
                if not class_results:
                    continue

                class_stabilities = []

                for result in class_results:
                    sample_idx  = result["sample_idx"]
                    pp_original = result["PP"]

                    # On ne peut calculer la stabilité que si PP existe
                    if pp_original is None:
                        continue

                    sample       = self.X_test.iloc[[sample_idx]].values.astype(np.float32)
                    feature_std  = self.X_train.std(axis=0).values.astype(np.float32)
                    correlations = []

                    for _ in range(n_perturbations):
                        try:
                            # Ajouter un petit bruit sur les features
                            noise     = np.random.normal(0, feature_std * 0.001, sample.shape)
                            perturbed = (sample + noise).astype(np.float32)

                            # Régénérer PP sur l'échantillon perturbé
                            cem_pp = CEM(
                                self._get_mlp_predict_fn,
                                mode='PP',
                                shape=(1, self.n_features),
                                kappa=0.1, beta=0.1, gamma=100,
                                ae_model=self._get_ae_predict_fn,
                                max_iterations=200,  # moins d'iterations pour aller plus vite
                                learning_rate_init=0.01,
                                feature_range=self.feature_range,
                                verbose=False
                            )
                            cem_pp.fit(
                                self.X_train.values.astype(np.float32),
                                no_info_type='median'
                            )
                            exp_perturbed = cem_pp.explain(perturbed)

                            if exp_perturbed.PP is not None:
                                pp_perturbed = exp_perturbed.PP.flatten()

                                # Corrélation entre PP original et PP perturbé
                                if (
                                    np.var(pp_original)  > 1e-10
                                    and np.var(pp_perturbed) > 1e-10
                                ):
                                    corr = np.corrcoef(pp_original, pp_perturbed)[0, 1]
                                    if not np.isnan(corr):
                                        correlations.append(max(0, corr))

                        except Exception as e:
                            print(f"      Erreur perturbation : {e}")
                            continue

                    if correlations:
                        class_stabilities.append(np.mean(correlations))

                if class_stabilities:
                    model_stab[class_name] = {
                        "mean"      : np.mean(class_stabilities),
                        "std"       : np.std(class_stabilities),
                        "individual": class_stabilities
                    }
                    print(
                        f"      {class_name} : "
                        f"{np.mean(class_stabilities):.4f} ± {np.std(class_stabilities):.4f}"
                    )

            stability_results[model_name] = model_stab

        self.cem_metrics["stability"] = stability_results
        print("\n   Stabilité calculée")
        return stability_results

    def calculate_fidelity(self):
        """
        Calcule la fidélité des explications CEM.

        Principe : le PP doit maintenir la même prédiction que l'échantillon original.
        On passe le PP dans le MLP wrapper et on vérifie que la classe
        prédite est identique à celle du modèle sklearn sur l'original.

        Fidélité = proportion d'échantillons où la classe est maintenue.
        1.0 = parfait, 0.0 = le PP ne maintient jamais la prédiction.
        """
        print("\n CALCUL DE LA FIDÉLITÉ CEM")
        print("=" * 60)

        fidelity_results = {}

        for model_name, mlp in self.mlp_wrappers.items():
            print(f"\n   Modèle : {model_name}")
            model_fidelities = {}

            for class_name, class_results in self.cem_results.get(model_name, {}).items():
                if not class_results:
                    continue

                class_fidelities = []

                for result in class_results:
                    pp_values = result["PP"]
                    if pp_values is None:
                        continue

                    try:
                        sample_idx    = result["sample_idx"]
                        sklearn_model = self.trained_models[model_name]

                        # Prédiction originale du vrai modèle sklearn
                        original_pred = int(sklearn_model.predict(
                            self.X_test.iloc[[sample_idx]]
                        )[0])

                        # Prédiction du MLP sur le PP
                        pp_tensor = torch.FloatTensor(pp_values.reshape(1, -1))
                        with torch.no_grad():
                            pp_probs = mlp(pp_tensor).numpy()[0]
                        pp_pred = int(np.argmax(pp_probs))

                        # 1.0 si même classe, 0.0 sinon
                        fidelity = 1.0 if pp_pred == original_pred else 0.0
                        class_fidelities.append(fidelity)

                    except Exception as e:
                        print(f"      Erreur fidélité échantillon {result['sample_idx']} : {e}")
                        continue

                if class_fidelities:
                    model_fidelities[class_name] = {
                        "mean"      : np.mean(class_fidelities),
                        "std"       : np.std(class_fidelities),
                        "individual": class_fidelities
                    }
                    print(
                        f"      {class_name} : "
                        f"{np.mean(class_fidelities):.4f} ± {np.std(class_fidelities):.4f}"
                    )

            fidelity_results[model_name] = model_fidelities

        self.cem_metrics["fidelity"] = fidelity_results
        print("\n   Fidélité calculée")
        return fidelity_results

    def print_summary(self):
        """
        Affiche un résumé complet des métriques CEM :
        stabilité et fidélité par classe et par modèle,
        avec évaluation qualitative.
        Affiche aussi les top features PP et PN par modèle.
        """
        print("\n" + "=" * 70)
        print(" RÉSUMÉ DES MÉTRIQUES CEM")
        print("=" * 70)
    
        stability = self.cem_metrics.get("stability", {})
        fidelity  = self.cem_metrics.get("fidelity", {})
    
        for model_name in self.trained_models.keys():
            if model_name not in self.cem_results:
                continue
    
            print(f"\n {model_name.upper().replace('_', ' ')}")
            print("-" * 60)
    
            # Stabilité
            print("   STABILITÉ (max = 1.0) :")
            stability_scores = []
            if model_name in stability:
                for class_name, data in stability[model_name].items():
                    print(f"      {class_name:8} : {data['mean']:.4f} ± {data['std']:.4f}")
                    stability_scores.append(data["mean"])
                overall_stability = np.mean(stability_scores) if stability_scores else 0
                print(f"      Overall  : {overall_stability:.4f}")
            else:
                print("      Pas de données")
                overall_stability = 0
    
            # Fidélité
            print("\n   FIDÉLITÉ (max = 1.0) :")
            fidelity_scores = []
            if model_name in fidelity:
                for class_name, data in fidelity[model_name].items():
                    print(f"      {class_name:8} : {data['mean']:.4f} ± {data['std']:.4f}")
                    fidelity_scores.append(data["mean"])
                overall_fidelity = np.mean(fidelity_scores) if fidelity_scores else 0
                print(f"      Overall  : {overall_fidelity:.4f}")
            else:
                print("      Pas de données")
                overall_fidelity = 0
    
            # Évaluation qualitative
            print("\n   ÉVALUATION :")
            stab_label = "élevée ✓"    if overall_stability > 0.8 \
                         else "modérée ⚠️" if overall_stability > 0.6 \
                         else "faible ✗"
            fid_label  = "élevée ✓"    if overall_fidelity > 0.8 \
                         else "modérée ⚠️" if overall_fidelity > 0.6 \
                         else "faible ✗"
            print(f"      Stabilité : {stab_label}")
            print(f"      Fidélité  : {fid_label}")
    
            # Top features PP et PN
            pp_counts = {}
            pn_counts = {}
            for class_results in self.cem_results[model_name].values():
                for result in class_results:
                    if result["PP_features"]:
                        for feat in result["PP_features"]:
                            pp_counts[feat] = pp_counts.get(feat, 0) + 1
                    if result["PN_features"]:
                        for feat in result["PN_features"]:
                            pn_counts[feat] = pn_counts.get(feat, 0) + 1
    
            print("\n   TOP 5 FEATURES PP (les plus fréquentes) :")
            for feat, count in sorted(
                pp_counts.items(), key=lambda x: x[1], reverse=True
            )[:5]:
                print(f"      {feat:40} : {count} fois")
    
            print("\n   TOP 5 FEATURES PN (les plus fréquentes) :")
            for feat, count in sorted(
                pn_counts.items(), key=lambda x: x[1], reverse=True
            )[:5]:
                print(f"      {feat:40} : {count} fois")


    def run(self, epochs=100, kappa=0.1, beta=0.1, gamma=100, max_iterations=500):
        """
        Lance tout le pipeline CEM dans l'ordre :
        1. Sélectionner les échantillons (RF + XGBoost consensus)
        2. Construire les MLP wrappers (un par modèle)
        3. Construire l'autoencoder (commun à tous les modèles)
        4. Générer les explications PP et PN
        5. Visualiser les résultats par modèle
        6. Calculer la stabilité
        7. Calculer la fidélité
        8. Afficher le résumé
    
        Args:
            epochs        : epochs pour MLP wrappers et autoencoder
            kappa         : marge de confiance CEM
            beta          : poids régularisation L1
            gamma         : poids autoencoder
            max_iterations: iterations max par optimisation CEM
        """
        print(" LANCEMENT DU PIPELINE CEM")
        print("=" * 70)
    
        # Étape 1 — Sélection des échantillons
        self.select_samples()
    
        # Étape 2 — MLP Wrappers
        self.build_mlp_wrappers(epochs=epochs)
    
        # Étape 3 — Autoencoder
        self.build_autoencoder(epochs=epochs)
    
        # Étape 4 — Explications CEM
        self.generate_cem_explanations(
            kappa=kappa,
            beta=beta,
            gamma=gamma,
            max_iterations=max_iterations
        )
    
        # Étape 5 — Visualisation par modèle
        print("\n GÉNÉRATION DES GRAPHIQUES PAR MODÈLE")
        print("=" * 70)
        for model_name in self.cem_results.keys():
            self.plot_cem_explanations(model_name)
    
        # Étape 6 — Stabilité
        self.calculate_stability()
    
        # Étape 7 — Fidélité
        self.calculate_fidelity()
    
        # Étape 8 — Résumé
        self.print_summary()
    
        print("\n PIPELINE CEM TERMINÉ !")

In [ ]:
cem = CEMEvaluator(evaluator)
cem.run()

In [7]:
mlp = MLPWrapper(184, 4)
print(mlp)

MLPWrapper(
  (network): Sequential(
    (0): Linear(in_features=184, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=4, bias=True)
    (7): Softmax(dim=1)
  )
)
